# Module 14 — KV Cache Optimization: PagedAttention, GQA, MQA

Last module we showed the KV cache is a memory monster. A Llama-2 70B at 4K context eats ~10 GB *just for the cache* of a single request. If you want to serve 50 users concurrently, the math stops working pretty fast.

This module is about the algorithmic tricks that made long-context serving actually viable. Three families, three very different ideas:

1. **PagedAttention** — borrow virtual memory from operating systems. Stop pre-allocating contiguous worst-case buffers.
2. **GQA / MQA** — stop storing K and V for every single attention head. Share them.
3. **Sliding Window Attention** — stop attending to every past token. Just look at the recent W.

We'll build toy simulators for each. Warning: we're going to wave our hands at some kernel details — the point is the *memory model*, not the CUDA.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
import math

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'monospace'
print('torch', torch.__version__)

---
## 14a — PagedAttention (vLLM's killer trick)

### The problem

Pre-PagedAttention, KV cache was allocated like this: for each new request, reserve a contiguous buffer of size `max_seq_len × d`. Maybe you set `max_seq_len = 4096`. A user sends a 50-token prompt and you generate 30 tokens. You used 80 slots. You *reserved* 4096. **98% of the buffer is grey emptiness.**

Multiply by 20 concurrent users. Your GPU is ~98% wasted and you can't fit a 21st request. This is internal fragmentation — literally the same problem OS designers solved in the 1960s.

### The OS analogy (this is the whole module)

| OS concept | PagedAttention |
|---|---|
| Virtual memory | Logical per-request KV sequence |
| Physical pages (4 KB) | GPU memory blocks (16 tokens of KV) |
| Page table | Block table (per-request list of block IDs) |
| Page fault | Sequence needs one more token than current blocks hold → allocate a new block |
| Copy-on-write | Shared system prompt prefix: one physical block, many block tables pointing at it |

That's it. vLLM did not invent a new algorithm. They asked "what if KV cache worked like `malloc` instead of `alloca`?"


### The math

**Traditional** allocation per request:
$$
\text{alloc}_{trad} = \text{max\_len} \cdot d \cdot 2 \cdot b
$$
where $d$ is hidden dim, $2$ is for K and V, $b$ is bytes per element. **Waste** per request:
$$
\text{waste}_{trad} = (\text{max\_len} - \text{actual\_len}) \cdot d \cdot 2 \cdot b
$$

**Paged** allocation with block size $B$ (tokens per block):
$$
\text{alloc}_{paged} = \left\lceil \frac{\text{actual\_len}}{B} \right\rceil \cdot B \cdot d \cdot 2 \cdot b
$$
Waste is at most one partial block:
$$
\text{waste}_{paged} \leq (B - 1) \cdot d \cdot 2 \cdot b
$$

With $B=16$ and realistic sequence length distributions, measured savings are **50–70%**. That's the whole "vLLM is 10x faster" story — they're not computing faster, they're fitting 10x more requests on the same GPU.


Let's plug in real numbers so the problem is concrete, not abstract.

In [ ]:
# Concrete numbers: let's plug in some values and see the waste.
d_model = 4096
bytes_per = 2       # fp16
max_len = 4096
actual_len = 128    # typical short request
B = 16              # block size

def bytes_human(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} TB'

trad = max_len * d_model * 2 * bytes_per
trad_waste = (max_len - actual_len) * d_model * 2 * bytes_per

blocks_needed = math.ceil(actual_len / B)
paged = blocks_needed * B * d_model * 2 * bytes_per
paged_waste = (blocks_needed * B - actual_len) * d_model * 2 * bytes_per

print(f'Request: actual_len={actual_len}, max_len={max_len}, d={d_model}')
print(f'  Traditional alloc: {bytes_human(trad):>10}  waste: {bytes_human(trad_waste):>10}  ({100*trad_waste/trad:.1f}%)')
print(f'  Paged alloc      : {bytes_human(paged):>10}  waste: {bytes_human(paged_waste):>10}  ({100*paged_waste/paged:.1f}%)')
print(f'  Savings: {bytes_human(trad - paged)}')

See that? For a short request, traditional allocation is ~32x bigger than what's actually used. Paged is within one block of perfect.

Now let's do this for a *realistic* server, not just one request.

### Simulator: 20 concurrent requests

Let's make this visual. We simulate 20 concurrent chat-like requests with a realistic length distribution, then draw what the GPU memory looks like under the two allocation strategies.

In [ ]:
# Simulator: 20 concurrent requests with a realistic length distribution.
# Chat workloads are heavy-tailed: most requests are short, a few are long.

N_REQ = 20
MAX_LEN = 512        # total slots per request in traditional mode
BLOCK = 16           # tokens per block in paged mode
TOTAL_SLOTS = N_REQ * MAX_LEN   # total GPU "cells" we'll draw

rng = np.random.default_rng(42)
# Mixture: 70% short (20-80 tokens), 25% medium (100-250), 5% long (400-500)
lens = []
for _ in range(N_REQ):
    r = rng.random()
    if r < 0.70:    lens.append(int(rng.integers(20, 80)))
    elif r < 0.95:  lens.append(int(rng.integers(100, 250)))
    else:           lens.append(int(rng.integers(400, 500)))
lens = np.array(lens)
print('request lengths:', lens)
print(f'mean={lens.mean():.0f}, median={np.median(lens):.0f}, max={lens.max()}')

Now we build two grids side by side: the traditional layout where every request reserves `MAX_LEN` slots, and a paged layout where each request allocates blocks of `BLOCK` tokens from a shared pool.

First, the traditional grid. One row per request, the whole row is "reserved" even if only a prefix is used.

In [ ]:
# Traditional: each request owns a full row of MAX_LEN.
# 0 = unallocated, 1 = used, 2 = wasted (reserved but empty)
trad_grid = np.full((N_REQ, MAX_LEN), 2, dtype=int)   # start everything as reserved-but-wasted
for i, L in enumerate(lens):
    trad_grid[i, :L] = 1                              # mark the actually-used prefix

total_used_trad  = (trad_grid == 1).sum()
total_waste_trad = (trad_grid == 2).sum()
print(f'Traditional: used={total_used_trad:5d}  wasted={total_waste_trad:5d}  '
      f'({100*total_waste_trad/TOTAL_SLOTS:.1f}% waste)')

Now the paged version. We maintain a cursor over a flat pool of blocks, and each request grabs blocks on demand — exactly like `malloc`. The `block_tables` list is our per-request "page table."  

In [ ]:
# Paged: pool of blocks. Each request allocates ceil(L/BLOCK) blocks.
paged_grid = np.zeros((N_REQ, MAX_LEN), dtype=int)    # 0=free, 1=used, 2=partial-block tail waste

block_cursor = 0        # next free block in the linear pool
block_tables = []       # per-request list of block ids (the "page table")

def block_coords(block_id):
    # map linear block id -> (row, col) in the paged_grid picture
    slots_per_row = MAX_LEN
    total_slot = block_id * BLOCK
    return total_slot // slots_per_row, total_slot % slots_per_row

for i, L in enumerate(lens):
    n_blocks_needed = math.ceil(L / BLOCK)
    table = []
    tokens_left = L
    for _ in range(n_blocks_needed):
        bid = block_cursor
        block_cursor += 1
        table.append(bid)
        r, c = block_coords(bid)
        used_in_block = min(BLOCK, tokens_left)
        tokens_left -= used_in_block
        paged_grid[r, c:c+used_in_block] = 1
        if used_in_block < BLOCK:
            paged_grid[r, c+used_in_block:c+BLOCK] = 2   # partial-block waste
    block_tables.append(table)

total_used_paged  = (paged_grid == 1).sum()
total_waste_paged = (paged_grid == 2).sum()
total_free_paged  = (paged_grid == 0).sum()
print(f'Paged: used={total_used_paged:5d}  partial-waste={total_waste_paged:5d}  free={total_free_paged:5d}')
print(f'Paged effective waste (reserved but unused): '
      f'{100*total_waste_paged/(total_used_paged+total_waste_paged):.1f}%')
print(f'Free capacity for MORE requests: {total_free_paged} slots '
      f'({100*total_free_paged/TOTAL_SLOTS:.1f}% of pool)')

### The picture

This is the part worth screenshotting.

In [ ]:
# Visualize both grids side by side.
# color map: 0=free(white), 1=used(green), 2=waste(grey)
cmap = ListedColormap(['#ffffff', '#3fa34d', '#c8c8c8'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].imshow(trad_grid, cmap=cmap, vmin=0, vmax=2, aspect='auto', interpolation='nearest')
axes[0].set_title(f'Traditional KV cache\n{100*total_waste_trad/TOTAL_SLOTS:.0f}% of GPU memory wasted (grey)')
axes[0].set_xlabel('token slot')
axes[0].set_ylabel('request id')

axes[1].imshow(paged_grid, cmap=cmap, vmin=0, vmax=2, aspect='auto', interpolation='nearest')
axes[1].set_title(f'PagedAttention\n{100*total_free_paged/TOTAL_SLOTS:.0f}% FREE for new requests')
axes[1].set_xlabel('slot in block pool')
axes[1].set_ylabel('pool row')

from matplotlib.patches import Patch
legend = [Patch(facecolor='#3fa34d', label='used'),
          Patch(facecolor='#c8c8c8', label='wasted / partial'),
          Patch(facecolor='#ffffff', edgecolor='#333', label='free (paged only)')]
axes[1].legend(handles=legend, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

Left grid: every request has reserved a giant strip. Look at request 0 — a few green pixels, then a sea of grey. That grey is memory you *cannot* give to anyone else. It's fragmented.

Right grid: requests tightly pack into blocks. The huge white area on the right is *free space*, which means new requests can land there. Same GPU, same pool — several times the throughput.

### Peeking at the page table

This is *the* key data structure in vLLM. Per request, a tiny list of physical block IDs. When you generate a new token and run out of room in the current block, you grab another block from the free list and append its ID. No large contiguous allocation required, ever.

In [ ]:
# Page tables for a few requests
for i in [0, 1, 2, 10, 19]:
    print(f'req {i:2d} (len={lens[i]:3d}): block_table = {block_tables[i]}')

Short requests get one or two block IDs. Long requests get a handful. The total memory taken is *proportional to actual use*, not to `max_seq_len`.

### How much does it depend on the workload?

Traditional waste is awful when most requests are short. What if requests are more uniform? Let's sweep.

In [ ]:
# Sweep: how does savings scale with pool utilization?
# We'll vary mean request length and replot waste fraction.

def simulate(n_req=50, max_len=512, block=16, short_bias=0.7, seed=0):
    rng = np.random.default_rng(seed)
    lens = []
    for _ in range(n_req):
        r = rng.random()
        if r < short_bias:   lens.append(int(rng.integers(20, 80)))
        elif r < 0.95:       lens.append(int(rng.integers(100, 250)))
        else:                lens.append(int(rng.integers(400, 500)))
    lens = np.array(lens)
    # traditional
    trad_reserved = n_req * max_len
    trad_used = lens.sum()
    trad_waste = (trad_reserved - trad_used) / trad_reserved
    # paged
    paged_reserved = sum(math.ceil(L/block) * block for L in lens)
    paged_waste = (paged_reserved - trad_used) / paged_reserved
    return trad_waste, paged_waste

biases = np.linspace(0.1, 0.95, 18)
trads, pageds = [], []
for b in biases:
    t, p = simulate(n_req=64, short_bias=b, seed=7)
    trads.append(t); pageds.append(p)

plt.figure(figsize=(8,4.5))
plt.plot(biases, np.array(trads)*100, 'o-', label='traditional', color='#888')
plt.plot(biases, np.array(pageds)*100, 's-', label='paged (B=16)', color='#3fa34d')
plt.xlabel('fraction of requests that are short')
plt.ylabel('fraction of allocated KV cache wasted (%)')
plt.title('PagedAttention vs traditional — waste across workload shapes')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Copy-on-write and the shared prefix trick

One more piece of OS magic: if 20 users send requests that all start with the same 2000-token system prompt, traditional serving recomputes (and re-stores) the KV for that prefix 20 times. With paged + COW, the prefix gets computed once, stored in a block set, and all 20 block tables just *point at it*. When a request starts writing its own tokens, it gets a fresh block — classic copy-on-write.

At 2000 tokens × 20 users × `2·n_layers·d_model·2 bytes`, on a 70B model, this is tens of GB saved. Free lunch.


Before we leave 14a, one more demo — the copy-on-write sharing. We just build fake block tables for 4 requests that share the same prefix, then diverge.

In [ ]:
# Toy COW visualization: 4 requests share a 3-block prefix, then diverge.
prefix_blocks = [101, 102, 103]  # physical ids
tables = {
    'req A': prefix_blocks + [201],
    'req B': prefix_blocks + [202, 203],
    'req C': prefix_blocks + [204],
    'req D': prefix_blocks + [205, 206, 207],
}
print('physical blocks 101-103 are shared system prompt. Each request just references them.\n')
for name, t in tables.items():
    arrow = ' -> '.join(f'[{b}]' for b in t)
    print(f'{name}: {arrow}')
print('\nOne physical copy of the prefix. Four logical views. That is COW.')

This trick, combined with paged allocation, is why modern serving can support thousands of users sharing a long system prompt at essentially zero extra KV cost per user.

### Takeaway for 14a

PagedAttention is less an attention algorithm and more a **memory allocator**. It's the OS design playbook — virtual memory, page tables, COW — ported to GPU KV cache. The attention kernel itself just has to know how to walk a block table instead of a flat buffer.


---
## 14b — GQA and MQA: stop giving every head its own K and V

PagedAttention attacks *how* you store the cache. GQA/MQA attack *how much there is to store* in the first place.

### Standard Multi-Head Attention

In MHA with $h$ heads and head dim $d_k$:

- Each head has its own $Q_i, K_i, V_i$.
- KV cache size per token: $2 \cdot h \cdot d_k = 2 \cdot d_{\text{model}}$.
- For a 32-head model at $d_{\text{model}}=4096$: **8192 values per token per layer.**

### Multi-Query Attention (MQA, Shazeer 2019)

Radical: **all heads share one K and one V.** Queries are still per-head (so heads still specialize), but K and V are singletons.

$$
\text{KV}_{MQA} = 2 \cdot 1 \cdot d_k = \frac{2 \cdot d_{\text{model}}}{h}
$$

For 32 heads, that's **32× smaller KV cache.** Problem: quality drops a measurable amount — training becomes unstable, some tasks regress.

### Grouped-Query Attention (GQA, Ainslie 2023)

The sensible middle ground. Split heads into $g$ groups. Each group shares one $K, V$.

$$
\text{KV}_{GQA} = 2 \cdot g \cdot d_k = \frac{2 \cdot d_{\text{model}} \cdot g}{h}
$$

- $g = h$: plain MHA
- $g = 1$: MQA
- $g = 8$ for $h = 64$: Llama-2 70B's choice → **8× smaller KV cache** with ~0 quality loss.

Essentially every modern frontier open model uses GQA: Llama 2/3/4, Mistral, DeepSeek, Qwen. It's the free lunch we all took.


In [ ]:
# Visualize the sharing pattern: which heads share which KV?
def make_sharing(h, g):
    '''returns length-h array where entry i is the group id of head i'''
    heads_per_group = h // g
    return np.array([i // heads_per_group for i in range(h)])

h = 16
configs = [('MHA  (g=16)', make_sharing(h, 16)),
           ('GQA  (g=8)',  make_sharing(h, 8)),
           ('GQA  (g=4)',  make_sharing(h, 4)),
           ('GQA  (g=2)',  make_sharing(h, 2)),
           ('MQA  (g=1)',  make_sharing(h, 1))]

fig, axes = plt.subplots(len(configs), 1, figsize=(10, 6))
for ax, (name, share) in zip(axes, configs):
    # draw head row
    for i, gid in enumerate(share):
        color = plt.cm.tab20(gid / max(1, share.max()))
        ax.add_patch(patches.Rectangle((i, 0), 0.9, 1, color=color))
        ax.text(i+0.45, 0.5, f'Q{i}', ha='center', va='center', fontsize=7, color='white')
    # KV labels below
    # one KV per unique group
    unique = sorted(set(share))
    for gid in unique:
        xs = [i for i, x in enumerate(share) if x == gid]
        x_center = np.mean(xs) + 0.45
        ax.plot([x_center, x_center], [0, -0.3], color='black', lw=0.7)
        ax.text(x_center, -0.55, f'KV{gid}', ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))
    ax.set_xlim(-0.5, h+0.5)
    ax.set_ylim(-1, 1.2)
    ax.set_title(name, loc='left', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
plt.suptitle('Q heads (top) and their shared KV projections (bottom)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Memory savings as a function of group count.
# Fixed: h=64, d_k=128, seq=4096, n_layers=80, fp16 (Llama-2 70B-ish)
h, d_k, seq, n_layers, b = 64, 128, 4096, 80, 2

def kv_bytes(g):
    # per token: 2 (K,V) * g * d_k * b, then * seq * n_layers
    return 2 * g * d_k * b * seq * n_layers

groups = [1, 2, 4, 8, 16, 32, 64]
sizes = [kv_bytes(g) for g in groups]
mha_size = kv_bytes(h)
savings = [mha_size / s for s in sizes]

fig, ax = plt.subplots(figsize=(8,4.5))
bars = ax.bar([str(g) for g in groups], [s/1e9 for s in sizes],
              color=['#d94444' if g==1 else '#3fa34d' if g==8 else '#888' for g in groups])
ax.set_ylabel('KV cache size (GB)')
ax.set_xlabel('number of KV groups (g)')
ax.set_title(f'Llama-ish 70B at 4K context  ({h} heads, {n_layers} layers)')
for bar, g, red in zip(bars, groups, savings):
    label = f'{bar.get_height():.1f} GB\n({red:.0f}× vs MHA)'
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, label,
            ha='center', fontsize=8)
ax.text(0, sizes[0]/1e9 * 0.5, 'MQA', ha='center', color='white', fontweight='bold')
ax.text(3, sizes[3]/1e9 * 0.7, 'GQA-8\n(Llama-2 70B)', ha='center', color='white', fontweight='bold', fontsize=8)
ax.text(6, sizes[6]/1e9 * 0.9, 'MHA', ha='center', color='white', fontweight='bold')
plt.ylim(0, max(sizes)/1e9 * 1.15)
plt.tight_layout()
plt.show()

That red bar on the left (MQA) is the cheapest but the most aggressive. The green bar (GQA with 8 groups) is what Llama-2 70B actually ships. You give up ~nothing in quality and the cache is **8× smaller**. If you want to understand why modern long-context serving is possible at all, it's half "PagedAttention" and half "GQA."

Let's actually implement it in PyTorch.

The MQA row at the bottom collapses all 16 Q heads onto one `KV0`. The GQA rows in the middle are where modern models live — look at "GQA (g=8)": two Q heads share each KV projection. That's Llama-2 70B territory (with `h=64, g=8` rather than our toy `h=16, g=8`, same idea).

### Actually implementing it

In [ ]:
# A minimal GQA attention forward. The trick is in how we expand K,V to match Q.
class GQAAttention(torch.nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_kv_groups=2):
        super().__init__()
        assert n_heads % n_kv_groups == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv = n_kv_groups
        self.d_head = d_model // n_heads
        # Q: full n_heads worth. K, V: only n_kv_groups worth.
        self.wq = torch.nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.wk = torch.nn.Linear(d_model, n_kv_groups * self.d_head, bias=False)
        self.wv = torch.nn.Linear(d_model, n_kv_groups * self.d_head, bias=False)
        self.wo = torch.nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1,2)  # B,h,T,d
        k = self.wk(x).view(B, T, self.n_kv,    self.d_head).transpose(1,2)  # B,g,T,d
        v = self.wv(x).view(B, T, self.n_kv,    self.d_head).transpose(1,2)  # B,g,T,d
        # Repeat K,V so each group serves (n_heads/n_kv_groups) query heads.
        repeat = self.n_heads // self.n_kv
        k = k.repeat_interleave(repeat, dim=1)   # -> B,h,T,d
        v = v.repeat_interleave(repeat, dim=1)
        att = (q @ k.transpose(-1,-2)) / math.sqrt(self.d_head)
        # causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        att = att.masked_fill(mask, float('-inf'))
        w = torch.softmax(att, dim=-1)
        out = (w @ v).transpose(1,2).reshape(B, T, self.d_model)
        return self.wo(out)

# How much smaller is the KV *parameter* footprint?
def kv_params(n_heads, n_kv, d_model):
    d_head = d_model // n_heads
    return 2 * n_kv * d_head * d_model   # wk + wv

d_model, h = 256, 8
for g in [8, 4, 2, 1]:
    layer = GQAAttention(d_model, h, g)
    n = sum(p.numel() for p in layer.parameters())
    kv_p = kv_params(h, g, d_model)
    x = torch.randn(2, 16, d_model)
    y = layer(x)
    print(f'g={g}: kv_proj params={kv_p:5d}  total={n:5d}  out shape={tuple(y.shape)}')

Sanity check: GQA still works and the shapes match plain MHA. The *only* difference is that K and V projection matrices are smaller and we `repeat_interleave` them before the matmul. That's the entire patch. Every modern inference library has a version of this ~5-line change.

### Memory vs quality — the GQA sweet spot

In [ ]:
# Fake "quality vs memory" curve — illustrative only.
# In real papers: GQA-8 retains ~99% MHA quality, MQA drops 2-5%.
g_values = np.array([1, 2, 4, 8, 16, 32, 64])
# memory relative to MHA:
rel_mem = g_values / 64
# illustrative quality curve: sigmoid-ish, quick saturation
quality = 1 - 0.05 * np.exp(-g_values/2.0)

fig, ax1 = plt.subplots(figsize=(8,4.5))
color = '#d94444'
ax1.plot(g_values, rel_mem*100, 'o-', color=color, label='KV memory (% of MHA)')
ax1.set_xscale('log', base=2)
ax1.set_xlabel('groups (g)')
ax1.set_ylabel('KV memory % of MHA', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(g_values); ax1.set_xticklabels(g_values)

ax2 = ax1.twinx()
color2 = '#3fa34d'
ax2.plot(g_values, quality*100, 's-', color=color2, label='quality (illustrative)')
ax2.set_ylabel('retained quality %', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(90, 101)

plt.title('GQA: most of the memory win, almost none of the quality loss')
ax1.axvline(8, linestyle='--', color='gray', alpha=0.6)
ax1.text(8.3, 60, 'Llama-2 70B\nsweet spot', color='gray', fontsize=9)
plt.tight_layout()
plt.show()

Quality in the toy plot is fictional, don't cite it, but the *shape* is real: quality plateaus quickly as $g$ grows. By $g=8$ you've recovered almost all of MHA quality at a fraction of the memory. GQA is close to a free lunch — which is why it's universal.

### Combining with PagedAttention

The two tricks compose multiplicatively. PagedAttention makes allocation efficient; GQA shrinks what you're allocating. An 8× smaller KV cache that's also 60% less wasted = ~20× effective improvement over naive MHA. That's not marketing, that's arithmetic.


---
## 14c — Sliding Window Attention

Third angle of attack: **don't attend to every past token at all.** Attention is $O(n^2)$ because each query looks at every key. If we only let each query look at the last $W$ keys, it becomes $O(n \cdot W)$.

This isn't new — Longformer (2020) did it, BigBird did it, Mistral 7B made it cool again with $W=4096$.

### The math

- Full causal attention: $O(n^2)$ compute and memory.
- Sliding window with size $W$: $O(n \cdot W)$.

At $n=32768$, $W=4096$: **8× fewer operations, 8× less cache to keep around.**

### But doesn't this break long-range?

Here's the clever bit: attention is *stacked*. Layer 1 can see $W$ tokens back. Layer 2's queries look at layer 1's outputs, which already incorporated context from $W$ tokens back — so layer 2 effectively sees $2W$. After $L$ layers, the receptive field is:

$$
\text{effective receptive field} = W \cdot L
$$

Mistral: $W=4096$, $L=32$ → effective field $\approx 131{,}072$ tokens. That's how they get usable long context without $O(n^2)$ cost.

It's the same "depth gives you reach" argument as CNNs.


In [ ]:
# Draw three attention masks side by side: full causal, sliding window, Longformer hybrid.
n = 48
W = 8
global_tokens = [0, 1, 2]   # first 3 tokens are "global" in the hybrid

# Full causal
full = np.tril(np.ones((n, n)))
# Sliding window (still causal)
sliding = np.zeros((n, n))
for i in range(n):
    for j in range(max(0, i-W+1), i+1):
        sliding[i, j] = 1
# Longformer hybrid: sliding + global (global tokens attend everywhere, and everyone attends to them)
hybrid = sliding.copy()
for g in global_tokens:
    hybrid[g, :g+1] = 1       # causal for global row
    hybrid[g:, g] = 1         # everyone attends to this global col

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, m, title in zip(axes,
                         [full, sliding, hybrid],
                         [f'Full causal\nO(n²) = {int(full.sum())} cells',
                          f'Sliding window W={W}\nO(nW) = {int(sliding.sum())} cells',
                          f'Longformer hybrid (W={W}, 3 global)\n{int(hybrid.sum())} cells']):
    ax.imshow(m, cmap='Greens', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('key position')
    ax.set_ylabel('query position')
plt.tight_layout()
plt.show()

print(f'savings: sliding uses {100*sliding.sum()/full.sum():.0f}% of full, hybrid uses {100*hybrid.sum()/full.sum():.0f}%')

Sliding window (middle) keeps only a diagonal band — every row has at most $W$ green cells. Longformer (right) keeps the band *plus* a few "global" tokens (rows 0-2) that everyone can talk to. Good for e.g. `<CLS>` tokens, instructions, or retrieval keys.

Let's actually implement sliding-window attention in PyTorch.

In [ ]:
# Implement sliding-window attention, run it, sanity-check vs full attention.
def sliding_window_attention(q, k, v, W):
    '''q,k,v: (B, h, T, d). Returns (B, h, T, d).'''
    B, H, T, D = q.shape
    att = (q @ k.transpose(-1,-2)) / math.sqrt(D)  # (B,h,T,T)
    # causal mask
    causal = torch.triu(torch.ones(T, T, device=q.device), diagonal=1).bool()
    # sliding: forbid keys older than W-1 behind
    idx = torch.arange(T, device=q.device)
    window = (idx.unsqueeze(0) < (idx.unsqueeze(1) - W + 1))   # True where too old
    mask = causal | window
    att = att.masked_fill(mask, float('-inf'))
    w = torch.softmax(att, dim=-1)
    return w @ v, w

B, H, T, D = 1, 2, 24, 16
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
v = torch.randn(B, H, T, D)

out_full, w_full = sliding_window_attention(q, k, v, W=T)     # W=T degenerates to full causal
out_win,  w_win  = sliding_window_attention(q, k, v, W=6)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(w_full[0,0].detach().numpy(), cmap='viridis')
axes[0].set_title('Full causal attention weights')
axes[1].imshow(w_win[0,0].detach().numpy(), cmap='viridis')
axes[1].set_title('Sliding window (W=6) weights')
for a in axes:
    a.set_xlabel('key'); a.set_ylabel('query')
plt.tight_layout(); plt.show()

Notice how the sliding-window weight matrix on the right is mostly dark — we've actively forbidden most key positions. The surviving weights form a narrow diagonal band.

### Receptive field grows with depth

This is the part that always surprises people: even though each individual layer only looks $W$ tokens back, *stacking* gives you reach.

In [ ]:
# Receptive field growth by depth.
# Show how a sliding-window stack reaches further with each layer.
W_vals = [128, 512, 1024, 4096]
layers = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(8,4.5))
for W in W_vals:
    ax.plot(layers, W*layers, label=f'W={W}')
ax.axhline(131072, linestyle='--', color='gray', alpha=0.6)
ax.text(2, 135000, 'Mistral 7B target ~131K', color='gray')
ax.set_xlabel('number of layers')
ax.set_ylabel('effective receptive field (tokens)')
ax.set_title('Why stacking sliding-window layers still gets you long context')
ax.set_yscale('log')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Mistral's configuration hits the 131K target by depth alone. Whether your task can *actually use* information from 131K tokens back is a different question — in practice, sliding-window models perform best when long-range dependencies are rare (typical for chat, code completion) and struggle when they're critical (structured retrieval, needle-in-a-haystack tests).

### Cost growth: the wall vs the ramp

In [ ]:
# Compute/memory comparison: n^2 vs n*W as context grows.
n_vals = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536])
W = 4096
full_cost   = n_vals ** 2
sliding_cost = n_vals * W

plt.figure(figsize=(8,4.5))
plt.plot(n_vals, full_cost,   'o-', label='full attention  (n²)')
plt.plot(n_vals, sliding_cost, 's-', label=f'sliding window (n·W, W={W})')
plt.xscale('log', base=2); plt.yscale('log', base=2)
plt.xlabel('sequence length n')
plt.ylabel('attention cost (ops, log scale)')
plt.title('Sliding window stops the quadratic explosion')
plt.legend(); plt.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

---
## Putting it all together

Three different attacks on the same bottleneck, and they compose cleanly:

| Technique | What it attacks | Savings | Quality cost |
|---|---|---|---|
| **PagedAttention** | fragmentation of allocated memory | 50–70% | none |
| **GQA** (g=8) | K/V duplication across heads | 4–8× | ~0 |
| **MQA** (g=1) | same, maximally | 32× (for h=32) | measurable |
| **Sliding window** (W=4096) | quadratic scaling with context | $n/W$ | depends on task |

vLLM + GQA + sliding window is the baseline recipe for modern long-context serving. A 70B model serving 50 concurrent users at 32K context would have been impossible in 2022; today it's a single DGX node. This is why.

### Checkpoint quiz

1. Why is the fragmentation problem in KV cache called "internal" and not "external"?
2. Llama-2 70B has 64 heads and uses GQA with 8 groups. By what factor does this reduce KV cache memory compared to MHA?
3. Mistral uses sliding window $W=4096$ over 32 layers. What's the effective receptive field in tokens, and why doesn't it matter for most practical contexts?
4. PagedAttention's "copy-on-write" trick saves memory for which specific serving pattern?
5. Why does MQA hurt quality more than GQA, and why is "more groups" not always better?

Next module: Mixture of Experts — cutting compute instead of cutting memory.
